# Inferential Statistics for LLM vs Embedder Brain Encoding

This notebook adds a fully explicit inferential-statistics section to the project, **without permutation tests**.

It provides:
1. **Paired model comparison** (Embedder vs LLM) using paired t-tests.
2. **p-values** for all tested comparisons.
3. **Effect sizes** using Cohen's *d* for paired samples.
4. **Multiple-comparison correction** using FDR (Benjamini–Hochberg) when running many tests.

The notebook is designed to integrate with outputs from `voxel_encoding.ipynb`.


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from pathlib import Path


## 1) Configuration

Set the paths below to the arrays already produced by your pipeline.

Expected common shapes:
- `accs_qwen_exp2`: `(n_participants, n_layers)`
- `accs_emb_exp2`: `(n_participants, n_layers)`
- `accs_qwen_exp3`: `(n_participants, n_layers)`
- `accs_emb_exp3`: `(n_participants, n_layers)`

If these arrays are already in memory (from running `voxel_encoding.ipynb`), skip file loading and use them directly.


In [ ]:
# Optional: load from disk if saved previously.
# If arrays already exist in memory (e.g., from voxel_encoding.ipynb), skip this block.

RESULTS_DIR = Path('results')

# Primary filenames used in this repo (voxel_encoding.ipynb save block)
candidate_files = {
    'accs_qwen_exp2': [RESULTS_DIR / 'qwen_layer_performances_exp2.npy', RESULTS_DIR / 'accs_qwen_exp2.npy'],
    'accs_emb_exp2': [RESULTS_DIR / 'qwen_embedding_layer_performances_exp2.npy', RESULTS_DIR / 'accs_embedder_exp2.npy'],
    'accs_qwen_exp3': [RESULTS_DIR / 'qwen_layer_performances_exp3.npy', RESULTS_DIR / 'accs_qwen_exp3.npy'],
    'accs_emb_exp3': [RESULTS_DIR / 'qwen_embedding_layer_performances_exp3.npy', RESULTS_DIR / 'accs_embedder_exp3.npy'],
}

for var_name, paths in candidate_files.items():
    if var_name in globals():
        continue
    for path in paths:
        if path.exists():
            globals()[var_name] = np.load(path)
            print(f'Loaded {var_name} from {path}')
            break


In [ ]:
# Basic shape sanity checks
required = ['accs_qwen_exp2', 'accs_emb_exp2', 'accs_qwen_exp3', 'accs_emb_exp3']
missing = [name for name in required if name not in globals()]
if missing:
    raise ValueError(
        'Missing arrays: ' + str(missing) + '. Run voxel_encoding.ipynb save cells or define arrays in memory.'
    )

print('Shapes:')
for name in required:
    print(f'{name}:', globals()[name].shape)

assert accs_qwen_exp2.shape == accs_emb_exp2.shape
assert accs_qwen_exp3.shape == accs_emb_exp3.shape


## 2) Statistical helper functions


In [ ]:
def cohens_d_paired(x, y):
    """Cohen's d for paired samples."""
    x = np.asarray(x)
    y = np.asarray(y)
    diff = x - y
    return diff.mean() / diff.std(ddof=1)


def paired_t_with_effect(embed_scores, llm_scores, alternative="greater"):
    """
    Paired t-test and paired Cohen's d.
    alternative: 'greater' tests Embedder > LLM.
    """
    embed_scores = np.asarray(embed_scores)
    llm_scores = np.asarray(llm_scores)
    if embed_scores.shape != llm_scores.shape:
        raise ValueError("embed_scores and llm_scores must have same shape")

    try:
        t_stat, p_val = stats.ttest_rel(embed_scores, llm_scores, alternative=alternative)
    except TypeError:
        t_stat, p_two = stats.ttest_rel(embed_scores, llm_scores)
        if alternative == "greater":
            p_val = p_two / 2 if t_stat > 0 else 1 - (p_two / 2)
        elif alternative == "less":
            p_val = p_two / 2 if t_stat < 0 else 1 - (p_two / 2)
        else:
            p_val = p_two

    d = cohens_d_paired(embed_scores, llm_scores)
    return t_stat, p_val, d


## 3) Main analysis A: Compare models at each model's own peak layer

For each participant:
- LLM score = participant's score at **LLM global peak layer**
- Embedder score = participant's score at **Embedder global peak layer**

Then run paired t-test across participants.


In [ ]:
# Global peak layers based on participant-averaged curves
peak_qwen_exp2 = int(np.argmax(accs_qwen_exp2.mean(axis=0)))
peak_emb_exp2  = int(np.argmax(accs_emb_exp2.mean(axis=0)))
peak_qwen_exp3 = int(np.argmax(accs_qwen_exp3.mean(axis=0)))
peak_emb_exp3  = int(np.argmax(accs_emb_exp3.mean(axis=0)))

print("Peak layers")
print(f"Exp2 LLM={peak_qwen_exp2}, Embedder={peak_emb_exp2}")
print(f"Exp3 LLM={peak_qwen_exp3}, Embedder={peak_emb_exp3}")

exp2_llm_peak_scores = accs_qwen_exp2[:, peak_qwen_exp2]
exp2_emb_peak_scores = accs_emb_exp2[:, peak_emb_exp2]
exp3_llm_peak_scores = accs_qwen_exp3[:, peak_qwen_exp3]
exp3_emb_peak_scores = accs_emb_exp3[:, peak_emb_exp3]

rows = []
for exp_name, emb, llm in [
    ("Exp2", exp2_emb_peak_scores, exp2_llm_peak_scores),
    ("Exp3", exp3_emb_peak_scores, exp3_llm_peak_scores),
]:
    t, p, d = paired_t_with_effect(emb, llm, alternative="greater")
    rows.append({
        "comparison": f"{exp_name}: Embedder_peak > LLM_peak",
        "n": len(emb),
        "embedder_mean": emb.mean(),
        "llm_mean": llm.mean(),
        "mean_diff": (emb - llm).mean(),
        "t_stat": t,
        "p_value": p,
        "cohens_d_paired": d,
    })

peak_df = pd.DataFrame(rows)
peak_df


## 4) Main analysis B: Layer-wise paired tests + FDR correction

For each layer in each experiment:
- paired t-test across participants (Embedder > LLM)
- then FDR-correct p-values across all tested layers.


In [ ]:
def layerwise_paired_tests(accs_emb, accs_llm, experiment_name):
    n_layers = accs_emb.shape[1]
    rows = []
    pvals = []

    for layer in range(n_layers):
        emb = accs_emb[:, layer]
        llm = accs_llm[:, layer]
        t, p, d = paired_t_with_effect(emb, llm, alternative="greater")
        rows.append({
            "experiment": experiment_name,
            "layer": layer,
            "embedder_mean": emb.mean(),
            "llm_mean": llm.mean(),
            "mean_diff": (emb - llm).mean(),
            "t_stat": t,
            "p_value": p,
            "cohens_d_paired": d,
        })
        pvals.append(p)

    rows_df = pd.DataFrame(rows)
    reject, p_fdr, _, _ = multipletests(pvals, alpha=0.05, method="fdr_bh")
    rows_df["p_fdr_bh"] = p_fdr
    rows_df["sig_fdr_0.05"] = reject
    return rows_df

layer_exp2_df = layerwise_paired_tests(accs_emb_exp2, accs_qwen_exp2, "Exp2")
layer_exp3_df = layerwise_paired_tests(accs_emb_exp3, accs_qwen_exp3, "Exp3")

layerwise_df = pd.concat([layer_exp2_df, layer_exp3_df], ignore_index=True)
layerwise_df.head()


In [ ]:
print("Exp2 significant layers after FDR:")
display(layer_exp2_df[layer_exp2_df["sig_fdr_0.05"]][["layer", "p_value", "p_fdr_bh", "cohens_d_paired", "mean_diff"]])

print("Exp3 significant layers after FDR:")
display(layer_exp3_df[layer_exp3_df["sig_fdr_0.05"]][["layer", "p_value", "p_fdr_bh", "cohens_d_paired", "mean_diff"]])


## 5) Optional robustness: same-layer paired comparisons at predefined layers


In [ ]:
def paired_test_at_layer(accs_emb, accs_llm, layer, exp_name):
    emb = accs_emb[:, layer]
    llm = accs_llm[:, layer]
    t, p, d = paired_t_with_effect(emb, llm, alternative="greater")
    return {
        "experiment": exp_name,
        "layer": layer,
        "embedder_mean": emb.mean(),
        "llm_mean": llm.mean(),
        "mean_diff": (emb - llm).mean(),
        "t_stat": t,
        "p_value": p,
        "cohens_d_paired": d,
    }

example_layers = [10, 16, 17, 25]
rows = []
for L in example_layers:
    if L < accs_emb_exp2.shape[1]:
        rows.append(paired_test_at_layer(accs_emb_exp2, accs_qwen_exp2, L, "Exp2"))
    if L < accs_emb_exp3.shape[1]:
        rows.append(paired_test_at_layer(accs_emb_exp3, accs_qwen_exp3, L, "Exp3"))

pd.DataFrame(rows)


## 6) Save statistics tables for report integration


In [ ]:
out_dir = Path("results/statistics")
out_dir.mkdir(parents=True, exist_ok=True)

peak_df.to_csv(out_dir / "paired_peak_tests.csv", index=False)
layerwise_df.to_csv(out_dir / "paired_layerwise_tests_fdr.csv", index=False)

print("Saved:")
print(out_dir / "paired_peak_tests.csv")
print(out_dir / "paired_layerwise_tests_fdr.csv")


## 7) Suggested report wording (copy-paste scaffold)


In [ ]:
report_text = """
We compared Qwen3-Embedding and Qwen3 using paired t-tests across participants.
For each experiment, we tested model differences at each model's peak layer and also layer-wise across all layers.
All tests were one-sided (Embedder > LLM), and we report p-values and paired Cohen's d effect sizes.
When many layer-wise tests were performed, we corrected p-values using the Benjamini–Hochberg FDR procedure (q=0.05).
"""
print(report_text)
